In [ ]:

!pip install -q sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.3 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import faiss
import numpy as np
# 1. Sample dataset
documents = [
    """
    Machine learning is a branch of artificial intelligence that enables systems
    to learn patterns from data and improve their performance without being
    explicitly programmed. It is widely used in recommendation systems, fraud
    detection, and predictive analytics.
    """,

    """
    Deep learning is a subset of machine learning that uses neural networks with
    many layers. It has achieved state-of-the-art results in image recognition,
    speech processing, and natural language understanding.
    """,

    """
    Natural language processing, or NLP, focuses on enabling computers to
    understand, interpret, and generate human language. Applications include
    translation, chatbots, summarization, and sentiment analysis.
    """,

    """
    FAISS is a library developed for efficient similarity search and clustering
    of dense vectors. It is commonly used in semantic search and retrieval
    systems where fast nearest-neighbor lookup is needed.
    """,

    """
    Retrieval-augmented generation combines a language model with a retrieval
    system. Instead of relying only on model memory, it first fetches relevant
    documents and then uses them to generate better grounded responses.
    """
]
# 2. Load embedding model and tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 3. Chunking function: 200-500 tokens
# We'll use max_tokens=300 and overlap=50 by default
def chunk_text(text, max_tokens=300, overlap=50):
    """
    Split text into chunks of about max_tokens with overlap.
    Returns a list of text chunks.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

        if end >= len(tokens):
            break

        start += max_tokens - overlap

    return chunks
# 4. Chunk all documents
chunked_passages = []

for doc_id, doc in enumerate(documents):
    chunks = chunk_text(doc, max_tokens=300, overlap=50)
    for chunk_id, chunk in enumerate(chunks):
        chunked_passages.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks created:", len(chunked_passages))
print("\nExample chunk:\n")
print(chunked_passages[0]["text"])
# 5. Generate embeddings for all chunks
texts = [item["text"] for item in chunked_passages]
embeddings = embedder.encode(texts, convert_to_numpy=True)

# FAISS expects float32
embeddings = np.array(embeddings).astype("float32")
# 6. Store embeddings in FAISS
# Using IndexFlatL2 for exact search
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("\nFAISS index size:", index.ntotal)
# 7. Retrieval function
def retrieve(query, top_k=3):
    """
    Embed the query, search FAISS, and return top_k passages.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "score": float(distances[0][rank]),
            "doc_id": chunked_passages[idx]["doc_id"],
            "chunk_id": chunked_passages[idx]["chunk_id"],
            "text": chunked_passages[idx]["text"]
        })

    return results
# 8. Test retrieval
query = "What is semantic search and retrieval?"
results = retrieve(query, top_k=3)

print(f"\nQuery: {query}\n")
for r in results:
    print(f"Rank: {r['rank']}")
    print(f"Score: {r['score']}")
    print(f"Doc ID: {r['doc_id']}, Chunk ID: {r['chunk_id']}")
    print("Passage:", r["text"])
    print("-" * 80)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks created: 5

Example chunk:

machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and improve their performance without being explicitly programmed. it is widely used in recommendation systems, fraud detection, and predictive analytics.

FAISS index size: 5

Query: What is semantic search and retrieval?

Rank: 1
Score: 0.9107091426849365
Doc ID: 4, Chunk ID: 0
Passage: retrieval - augmented generation combines a language model with a retrieval system. instead of relying only on model memory, it first fetches relevant documents and then uses them to generate better grounded responses.
--------------------------------------------------------------------------------
Rank: 2
Score: 1.1187703609466553
Doc ID: 3, Chunk ID: 0
Passage: faiss is a library developed for efficient similarity search and clustering of dense vectors. it is commonly used in semantic search and retrieval systems where fast nearest - neighbor lookup is nee